In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e11/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e11/train.csv
/kaggle/input/competitions/playground-series-s6e11/test.csv


In [2]:
# Step 1: load the tables and look at what is inside
SKIP_SUPERSEDED = True  # Skip 0-weight superseded early models to save >90 mins of GPU quota

if "train" not in globals():
    d = "/kaggle/input/competitions/playground-series-s6e11/"
    train = pd.read_csv(d + "train.csv")
    test = pd.read_csv(d + "test.csv")
    print("rows, columns:", train.shape, test.shape)
    print("share of fraud rows:", train["is_fraud"].mean())

for _npz in ["preds_backup.npz", "preds_opt2.npz", "preds_exp3.npz", "preds_exp4.npz", "preds_exp5.npz"]:
    _p = "/kaggle/working/" + _npz
    if os.path.exists(_p):
        _b = np.load(_p)
        for _k in _b.files:
            if _k not in globals():
                globals()[_k] = _b[_k].astype(np.float64)
        print("Restored cache:", _npz, len(_b.files))


rows, columns: (699960, 26) (299984, 25)
share of fraud rows: 0.016699525687182123


In [3]:
# Step 2: Feature Engineering + 5-fold LightGBM
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def make_features(df):
    out = df.copy()
    out["amount_to_balance"] = out["amount_usd"] / (out["account_balance_usd"] + 1.0)
    out["balance_after_txn"] = out["account_balance_usd"] - out["amount_usd"]
    out["txn_burst_rate"] = out["txn_count_last_24h"] / (out["hours_since_last_txn"] + 0.1)
    flag_cols = [
        "is_foreign_transaction", "is_new_merchant", "used_vpn",
        "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
    ]
    out["risk_flag_count"] = out[flag_cols].sum(axis=1)
    out["amount_x_merchant_risk"] = out["amount_usd"] * out["merchant_risk_score"]
    return out

train_fe = make_features(train)
test_fe = make_features(test)

text_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for c in text_cols:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = pd.Categorical(test_fe[c], categories=train_fe[c].cat.categories)

X = train_fe.drop(columns=["id", "is_fraud"])
y = train_fe["is_fraud"]
X_test = test_fe.drop(columns=["id"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

if "oof_preds" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=fold, verbose=-1, n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / 5

if "oof_preds" not in globals():
    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
lgb_oof = oof_preds.copy()
lgb_test = test_preds.copy()

In [4]:
# Step 3: 5-fold XGBoost
import xgboost as xgb

if "xgb_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    xgb_oof = np.zeros(len(train_fe))
    xgb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        xgb_model = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=100,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            device="cuda",
            enable_categorical=True,
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=fold,
            n_jobs=-1,
        )
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        xgb_oof[va_idx] = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test += xgb_model.predict_proba(X_test)[:, 1] / 5

if "xgb_oof" not in globals():
    xgb_oof = np.zeros(len(train_fe))
    xgb_test = np.zeros(len(test_fe))

In [5]:
# Step 4: 5-fold CatBoost + 3-Model Blend (LightGBM + XGBoost + CatBoost)
from catboost import CatBoostClassifier

if "cb_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    X_cb = X.copy()
    X_test_cb = X_test.copy()
    for c in text_cols:
        X_cb[c] = X_cb[c].cat.codes
        X_test_cb[c] = X_test_cb[c].cat.codes

    cb_oof = np.zeros(len(train_fe))
    cb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y)):
        X_tr, y_tr = X_cb.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_cb.iloc[va_idx], y.iloc[va_idx]

        cb_model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.06,
            depth=6,
            cat_features=text_cols,
            one_hot_max_size=25,
            eval_metric="AUC",
            random_seed=fold,
            task_type="GPU",
            verbose=False,
        )
        cb_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=50,
            verbose=False,
        )

        cb_oof[va_idx] = cb_model.predict_proba(X_va)[:, 1]
        cb_test += cb_model.predict_proba(X_test_cb)[:, 1] / 5

if "cb_oof" not in globals():
    cb_oof = np.zeros(len(train_fe))
    cb_test = np.zeros(len(test_fe))
blend3_oof = np.zeros(len(train_fe))
blend3_test = np.zeros(len(test_fe))

In [6]:
# Step 5: Test 12 Non-Leaky Group Aggregation Z-Scores on 5-fold LightGBM
if "lgb_grp_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    def add_group_aggregations(tr_df, te_df):
        full = pd.concat([tr_df.drop(columns=["is_fraud"]), te_df], ignore_index=True)
        for grp in ["merchant_category", "channel", "auth_method"]:
            for num in ["amount_usd", "velocity_score", "merchant_risk_score", "distance_from_home_km"]:
                means = full.groupby(grp, observed=False)[num].transform("mean")
                stds = full.groupby(grp, observed=False)[num].transform("std") + 1e-5
                full[f"{num}_z_{grp}"] = ((full[num] - means) / stds).astype(np.float32)
        tr_out = full.iloc[:len(tr_df)].drop(columns=["id"]).copy()
        te_out = full.iloc[len(tr_df):].drop(columns=["id"]).copy().reset_index(drop=True)
        return tr_out, te_out
    
    X_grp, X_test_grp = add_group_aggregations(train_fe, test_fe)
    for c in text_cols:
        X_grp[c] = X_grp[c].astype("category")
        X_test_grp[c] = pd.Categorical(X_test_grp[c], categories=X_grp[c].cat.categories)
    
    lgb_grp_oof = np.zeros(len(train_fe))
    lgb_grp_test = np.zeros(len(test_fe))
    grp_imp = np.zeros(X_grp.shape[1])
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_grp, y)):
        X_tr, y_tr = X_grp.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_grp.iloc[va_idx], y.iloc[va_idx]
    
        model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=fold, verbose=-1, n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        lgb_grp_oof[va_idx] = model.predict_proba(X_va)[:, 1]
        lgb_grp_test += model.predict_proba(X_test_grp)[:, 1] / 5
        grp_imp += model.feature_importances_ / 5
        print(f"LGB+GroupAgg Fold {fold} AUC: {roc_auc_score(y_va, lgb_grp_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    grp_auc = roc_auc_score(y, lgb_grp_oof)
    print(f"\nLightGBM baseline OOF AUC:    {roc_auc_score(y, lgb_oof):.5f}")
    print(f"LightGBM + GroupAgg OOF AUC:  {grp_auc:.5f} (diff: {grp_auc - roc_auc_score(y, lgb_oof):+.5f})")
    
    best_w, best_auc = 0.0, roc_auc_score(y, blend3_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
        cand = w * lgb_grp_oof + (1 - w) * blend3_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} LGB_Grp + {1-w:.1f} Prev_3Way): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    final_test = best_w * lgb_grp_test + (1 - best_w) * blend3_test
    print("\nTop 12 features:")
    print(pd.Series(grp_imp, index=X_grp.columns).sort_values(ascending=False).head(12).round(1))
if "lgb_grp_oof" not in globals():
    lgb_grp_oof = np.zeros(len(train_fe))
    lgb_grp_test = np.zeros(len(test_fe))


In [7]:
# Step 6: Digit-Level Features + Shallow Trees (num_leaves=4, max_depth=2)
def add_digit_features(df):
    out = df.copy()
    float_cols = [
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "velocity_score", "merchant_risk_score"
    ]
    for col in float_cols:
        v = out[col].fillna(0).abs()
        int_v = np.floor(v).astype(np.int64)
        cents = np.round((v - int_v) * 100).astype(np.int32)
        out[f"{col}_cents"] = cents
        out[f"{col}_tenth"] = (cents // 10).astype(np.int32)
        out[f"{col}_hundredth"] = (cents % 10).astype(np.int32)
        out[f"{col}_ones"] = (int_v % 10).astype(np.int32)
        out[f"{col}_tens"] = ((int_v // 10) % 10).astype(np.int32)
        out[f"{col}_log1p"] = np.log1p(v).astype(np.float32)
    return out

X_dig = add_digit_features(train_fe.drop(columns=["id", "is_fraud"]))
X_test_dig = add_digit_features(test_fe.drop(columns=["id"]))

for c in text_cols:
    X_dig[c] = X_dig[c].astype("category")
    X_test_dig[c] = pd.Categorical(X_test_dig[c], categories=X_dig[c].cat.categories)

if "lgb_shallow_oof" not in globals():
    lgb_shallow_oof = np.zeros(len(train_fe))
    lgb_shallow_test = np.zeros(len(test_fe))
    sh_imp = np.zeros(X_dig.shape[1])
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr, y_tr = X_dig.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_dig.iloc[va_idx], y.iloc[va_idx]
    
        model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.05,
            num_leaves=4,
            max_depth=2,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=fold,
            verbose=-1,
            n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        lgb_shallow_oof[va_idx] = model.predict_proba(X_va)[:, 1]
        lgb_shallow_test += model.predict_proba(X_test_dig)[:, 1] / 5
        sh_imp += model.feature_importances_ / 5
        print(f"Shallow+Digits Fold {fold} AUC: {roc_auc_score(y_va, lgb_shallow_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    sh_auc = roc_auc_score(y, lgb_shallow_oof)
    print(f"\nPrevious Deep LightGBM (31 leaves):  {roc_auc_score(y, lgb_oof):.5f}")
    print(f"Previous 4-Model Ensemble:           {roc_auc_score(y, 0.2 * lgb_grp_oof + 0.8 * blend3_oof):.5f}")
    print(f"NEW Shallow LGB (4 leaves) + Digits: {sh_auc:.5f} (vs deep LGB: {sh_auc - roc_auc_score(y, lgb_oof):+.5f})")
    
    prev_best_oof = 0.2 * lgb_grp_oof + 0.8 * blend3_oof
    prev_best_test = 0.2 * lgb_grp_test + 0.8 * blend3_test
    
    best_w, best_auc = 0.0, roc_auc_score(y, prev_best_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        cand = w * lgb_shallow_oof + (1 - w) * prev_best_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} Shallow_Digits + {1-w:.1f} Prev_Best): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    print(f"\nNew Best Ensemble OOF AUC: {best_auc:.5f}")
    final_test = best_w * lgb_shallow_test + (1 - best_w) * prev_best_test
    print("\nTop 12 features in Shallow+Digits:")
    print(pd.Series(sh_imp, index=X_dig.columns).sort_values(ascending=False).head(12).round(1))

Shallow+Digits Fold 0 AUC: 0.95173 (tree 1415)


KeyboardInterrupt: 

In [ ]:
# Step 7: Nested Out-of-Fold Target Encoding on Digit & Category Keys + Shallow Trees
te_keys = [
    "amount_usd_cents", "hours_since_last_txn_cents", "distance_from_home_km_cents",
    "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents",
    "amount_usd_ones", "amount_usd_tens", "hours_since_last_txn_ones",
    "merchant_category", "card_type", "auth_method", "channel", "device_type"
]

if "lgb_nte_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    lgb_nte_oof = np.zeros(len(train_fe))
    lgb_nte_test = np.zeros(len(test_fe))
    inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    smooth = 20.0
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr = X_dig.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx]
        X_va = X_dig.iloc[va_idx].copy()
        y_va = y.iloc[va_idx]
        X_te = X_test_dig.copy()
    
        global_prior = y_tr.mean()
        tr_te = pd.DataFrame(index=X_tr.index)
        va_te = pd.DataFrame(index=X_va.index)
        te_te = pd.DataFrame(index=X_te.index)
    
        for col in te_keys:
            col_name = f"nte_{col}"
            tr_vals = np.full(len(X_tr), global_prior, dtype=np.float32)
    
            # Nested inner folds: each row is encoded WITHOUT its own label!
            s_tr = X_tr[col].astype(str)
            for in_tr, in_va in inner_skf.split(X_tr, y_tr):
                stats = y_tr.iloc[in_tr].groupby(s_tr.iloc[in_tr]).agg(["count", "sum"])
                te_map = (stats["sum"] + smooth * global_prior) / (stats["count"] + smooth)
                tr_vals[in_va] = s_tr.iloc[in_va].map(te_map).fillna(global_prior).values
    
            tr_te[col_name] = tr_vals
    
            # Outer val & test use all of (X_tr, y_tr)
            stats_full = y_tr.groupby(s_tr).agg(["count", "sum"])
            te_map_full = (stats_full["sum"] + smooth * global_prior) / (stats_full["count"] + smooth)
            va_te[col_name] = X_va[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
            te_te[col_name] = X_te[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
    
        X_tr_full = pd.concat([X_tr, tr_te], axis=1)
        X_va_full = pd.concat([X_va, va_te], axis=1)
        X_te_full = pd.concat([X_te, te_te], axis=1)
    
        model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.04,
            num_leaves=3,
            max_depth=2,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=fold,
            verbose=-1,
            n_jobs=-1,
        )
        model.fit(
            X_tr_full, y_tr,
            eval_set=[(X_va_full, y_va)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        lgb_nte_oof[va_idx] = model.predict_proba(X_va_full)[:, 1]
        lgb_nte_test += model.predict_proba(X_te_full)[:, 1] / 5
        print(f"Shallow+Digits+NestedTE Fold {fold} AUC: {roc_auc_score(y_va, lgb_nte_oof[va_idx]):.5f} (tree {model.best_iteration_})")
    
    nte_auc = roc_auc_score(y, lgb_nte_oof)
    print(f"\nShallow LGB + Digits alone:           {roc_auc_score(y, lgb_shallow_oof):.5f}")
    print(f"NEW Shallow LGB + Digits + Nested TE: {nte_auc:.5f} (diff: {nte_auc - roc_auc_score(y, lgb_shallow_oof):+.5f})")
    
    prev_best_oof = 0.8 * lgb_shallow_oof + 0.2 * (0.2 * lgb_grp_oof + 0.8 * blend3_oof)
    prev_best_test = 0.8 * lgb_shallow_test + 0.2 * (0.2 * lgb_grp_test + 0.8 * blend3_test)
    
    best_w, best_auc = 0.0, roc_auc_score(y, prev_best_oof)
    for w in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
        cand = w * lgb_nte_oof + (1 - w) * prev_best_oof
        cand_auc = roc_auc_score(y, cand)
        print(f"Blend ({w:.1f} NestedTE + {1-w:.1f} Prev_Best): {cand_auc:.5f}")
        if cand_auc > best_auc:
            best_auc, best_w = cand_auc, w
    
    print(f"\nNew Best Ensemble OOF AUC: {best_auc:.5f}")
    final_test = best_w * lgb_nte_test + (1 - best_w) * prev_best_test
if "lgb_nte_oof" not in globals():
    lgb_nte_oof = np.zeros(len(train_fe))
    lgb_nte_test = np.zeros(len(test_fe))


In [ ]:
# Step 8: Shallow XGBoost (max_depth=2) & Shallow CatBoost (depth=3) on Digits + Nested TE
from catboost import CatBoostClassifier
import xgboost as xgb

te_keys = [
    "amount_usd_cents", "hours_since_last_txn_cents", "distance_from_home_km_cents",
    "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents",
    "amount_usd_ones", "amount_usd_tens", "hours_since_last_txn_ones",
    "merchant_category", "card_type", "auth_method", "channel", "device_type"
]
inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smooth = 20.0

def get_fold_nte(tr_idx, va_idx):
    X_tr = X_dig.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]
    X_va = X_dig.iloc[va_idx].copy()
    X_te = X_test_dig.copy()
    global_prior = y_tr.mean()
    tr_te = pd.DataFrame(index=X_tr.index)
    va_te = pd.DataFrame(index=X_va.index)
    te_te = pd.DataFrame(index=X_te.index)
    for col in te_keys:
        col_name = f"nte_{col}"
        tr_vals = np.full(len(X_tr), global_prior, dtype=np.float32)
        s_tr = X_tr[col].astype(str)
        for in_tr, in_va in inner_skf.split(X_tr, y_tr):
            stats = y_tr.iloc[in_tr].groupby(s_tr.iloc[in_tr]).agg(["count", "sum"])
            te_map = (stats["sum"] + smooth * global_prior) / (stats["count"] + smooth)
            tr_vals[in_va] = s_tr.iloc[in_va].map(te_map).fillna(global_prior).values
        tr_te[col_name] = tr_vals
        stats_full = y_tr.groupby(s_tr).agg(["count", "sum"])
        te_map_full = (stats_full["sum"] + smooth * global_prior) / (stats_full["count"] + smooth)
        va_te[col_name] = X_va[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
        te_te[col_name] = X_te[col].astype(str).map(te_map_full).fillna(global_prior).astype(np.float32)
    return (
        pd.concat([X_tr, tr_te], axis=1),
        pd.concat([X_va, va_te], axis=1),
        pd.concat([X_te, te_te], axis=1),
    )

if "xgb_sh_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    xgb_sh_oof = np.zeros(len(train_fe))
    xgb_sh_test = np.zeros(len(test_fe))
    cb_sh_oof = np.zeros(len(train_fe))
    cb_sh_test = np.zeros(len(test_fe))
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig, y)):
        X_tr_f, X_va_f, X_te_f = get_fold_nte(tr_idx, va_idx)
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
        # 1. Shallow XGBoost (max_depth=2)
        xm = xgb.XGBClassifier(
            n_estimators=4000, learning_rate=0.05, max_depth=2,
            min_child_weight=100, subsample=0.8, colsample_bytree=0.8,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=100,
            random_state=fold, n_jobs=-1,
        )
        xm.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], verbose=False)
        xgb_sh_oof[va_idx] = xm.predict_proba(X_va_f)[:, 1]
        xgb_sh_test += xm.predict_proba(X_te_f)[:, 1] / 5
    
        # 2. Shallow CatBoost (depth=3)
        X_tr_cb, X_va_cb, X_te_cb = X_tr_f.copy(), X_va_f.copy(), X_te_f.copy()
        for c in text_cols:
            X_tr_cb[c] = X_tr_cb[c].cat.codes
            X_va_cb[c] = X_va_cb[c].cat.codes
            X_te_cb[c] = X_te_cb[c].cat.codes
    
        cm = CatBoostClassifier(
            iterations=3000, learning_rate=0.06, depth=3,
            cat_features=text_cols, one_hot_max_size=25,
            eval_metric="AUC", random_seed=fold, task_type="GPU", verbose=False,
        )
        cm.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), early_stopping_rounds=100, verbose=False)
        cb_sh_oof[va_idx] = cm.predict_proba(X_va_cb)[:, 1]
        cb_sh_test += cm.predict_proba(X_te_cb)[:, 1] / 5
    
        print(f"Fold {fold} | Shallow XGB: {roc_auc_score(y_va, xgb_sh_oof[va_idx]):.5f} (tree {xm.best_iteration}) | Shallow CB: {roc_auc_score(y_va, cb_sh_oof[va_idx]):.5f} (tree {cm.get_best_iteration()})")
    
    print("\n--- Shallow Models on Digits + Nested TE (5-fold OOF AUC) ---")
    print(f"Shallow LightGBM (3 leaves): {roc_auc_score(y, lgb_nte_oof):.5f}")
    print(f"Shallow XGBoost (depth=2):   {roc_auc_score(y, xgb_sh_oof):.5f}")
    print(f"Shallow CatBoost (depth=3):  {roc_auc_score(y, cb_sh_oof):.5f}")
    
    # Optimize 5-member ensemble (Shallow LGB+NTE, Shallow XGB+NTE, Shallow CB+NTE, Shallow LGB+Digits, Deep 3-Way)
    deep_oof = 0.2 * lgb_grp_oof + 0.8 * blend3_oof
    deep_test = 0.2 * lgb_grp_test + 0.8 * blend3_test
    
    from scipy.optimize import minimize
    models_oof = np.column_stack([lgb_nte_oof, xgb_sh_oof, cb_sh_oof, lgb_shallow_oof, deep_oof])
    models_test = np.column_stack([lgb_nte_test, xgb_sh_test, cb_sh_test, lgb_shallow_test, deep_test])
    
    def loss(weights):
        w = np.maximum(weights, 0)
        w = w / (w.sum() + 1e-12)
        return -roc_auc_score(y, models_oof @ w)
    
    res = minimize(loss, [0.3, 0.2, 0.2, 0.2, 0.1], method="Nelder-Mead")
    best_w = np.maximum(res.x, 0)
    best_w /= best_w.sum()
    names = ["Shallow_LGB_NTE", "Shallow_XGB_NTE", "Shallow_CB_NTE", "Shallow_LGB_Dig", "Deep_Blend"]
    for n, w in zip(names, best_w):
        print(f"  Weight {n:16s}: {w:.3f}")
    
    final_oof = models_oof @ best_w
    final_test = models_test @ best_w
    print(f"\nNEW 5-Member Ensemble OOF AUC: {roc_auc_score(y, final_oof):.5f}")
if "xgb_sh_oof" not in globals():
    xgb_sh_oof = np.zeros(len(train_fe))
    xgb_sh_test = np.zeros(len(test_fe))
    cb_sh_oof = np.zeros(len(train_fe))
    cb_sh_test = np.zeros(len(test_fe))
    deep_oof = np.zeros(len(train_fe))
    deep_test = np.zeros(len(test_fe))


In [ ]:
# Step 9: Finer Digit Keys + Frequency + Fast Nested TE + 2-Leaf Stumps & 3-Leaf Trees
def add_finer_digits(df):
    out = df.copy()
    float_cols = [
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "velocity_score", "merchant_risk_score"
    ]
    for col in float_cols:
        v = out[col].fillna(0).abs()
        int_v = np.floor(v).astype(np.int64)
        cents = np.round((v - int_v) * 100).astype(np.int32)
        out[f"{col}_int"] = int_v.astype(np.int32)
        out[f"{col}_div10"] = (int_v // 10).astype(np.int32)
        out[f"{col}_div100"] = (int_v // 100).astype(np.int32)
        out[f"{col}_hundreds"] = ((int_v // 100) % 10).astype(np.int32)
        out[f"{col}_thousands"] = ((int_v // 1000) % 10).astype(np.int32)
        out[f"{col}_is_round_cents"] = (cents == 0).astype(np.int8)
        out[f"{col}_is_99_cents"] = (cents == 99).astype(np.int8)
        # Leading digit (1..9, or 0 for 0.0) using fast vectorized log10
        v_pos = np.where(v > 0, v, 1.0)
        lead = np.floor(np.where(v > 0, v / (10.0 ** np.floor(np.log10(v_pos))), 0.0) + 1e-9)
        out[f"{col}_first_digit"] = np.clip(lead, 0, 9).astype(np.int8)
    for col in ["customer_age", "card_age_months"]:
        out[f"{col}_ones"] = (out[col] % 10).astype(np.int32)
        out[f"{col}_tens"] = (out[col] // 10).astype(np.int32)
    return out

if "X_dig2" not in globals():
    X_dig2 = add_finer_digits(X_dig)
    X_test_dig2 = add_finer_digits(X_test_dig)

    # Frequency encoding on key columns (train+test value counts, zero label leakage)
    n_tr = len(X_dig2)
    freq_cols = [
        "amount_usd_cents", "hours_since_last_txn_cents", "account_balance_usd_cents",
        "amount_usd_int", "hours_since_last_txn_int", "distance_from_home_km_int",
        "customer_age", "card_age_months", "merchant_risk_score_int", "velocity_score_int"
    ]
    for col in freq_cols:
        comb = pd.concat([X_dig2[col], X_test_dig2[col]], ignore_index=True)
        vc = comb.value_counts()
        X_dig2[f"{col}_freq"] = X_dig2[col].map(vc).astype(np.float32)
        X_test_dig2[f"{col}_freq"] = X_test_dig2[col].map(vc).astype(np.float32)

    # Pre-factorize all discrete & digit keys for ultra-fast np.bincount Nested TE
nte2_keys = [
    "merchant_category", "card_type", "auth_method", "channel", "device_type",
    "time_of_day_hour", "day_of_week", "cvv_retry_count", "prior_disputes",
    "customer_age", "card_age_months", "txn_count_last_24h",
    "amount_usd_cents", "amount_usd_tenth", "amount_usd_hundredth", "amount_usd_ones", "amount_usd_tens", "amount_usd_int", "amount_usd_div10", "amount_usd_div100", "amount_usd_first_digit",
    "hours_since_last_txn_cents", "hours_since_last_txn_ones", "hours_since_last_txn_tens", "hours_since_last_txn_int", "hours_since_last_txn_first_digit",
    "distance_from_home_km_cents", "distance_from_home_km_ones", "distance_from_home_km_tens", "distance_from_home_km_div10", "distance_from_home_km_first_digit",
    "account_balance_usd_cents", "account_balance_usd_ones", "account_balance_usd_tens", "account_balance_usd_div10", "account_balance_usd_div100", "account_balance_usd_first_digit",
    "velocity_score_cents", "velocity_score_ones", "velocity_score_tens", "velocity_score_int", "velocity_score_first_digit",
    "merchant_risk_score_cents", "merchant_risk_score_ones", "merchant_risk_score_tens", "merchant_risk_score_int", "merchant_risk_score_first_digit",
]

n_tr = len(X_dig2)
if "codes_tr_dict" not in globals():
    codes_tr_dict, codes_te_dict, ncats_dict = {}, {}, {}
    for col in nte2_keys:
        comb = pd.concat([X_dig2[col].astype(str), X_test_dig2[col].astype(str)], ignore_index=True)
        codes, _ = pd.factorize(comb, sort=False)
        codes_tr_dict[col] = codes[:n_tr]
        codes_te_dict[col] = codes[n_tr:]
        ncats_dict[col] = int(codes.max()) + 1

inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smooth = 20.0
y_arr = y.values.astype(np.float64)

if "lgb_dig2_oof" not in globals() and not globals().get("SKIP_SUPERSEDED", False):
    lgb_stump_oof = np.zeros(n_tr)
    lgb_stump_test = np.zeros(len(X_test_dig2))
    lgb_dig2_oof = np.zeros(n_tr)
    lgb_dig2_test = np.zeros(len(X_test_dig2))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        global_prior = y_tr_arr.mean()

        tr_te, va_te, te_te = {}, {}, {}
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        for col in nte2_keys:
            c_all_tr = codes_tr_dict[col]
            c_tr = c_all_tr[tr_idx]
            c_va = c_all_tr[va_idx]
            c_te = codes_te_dict[col]
            nc = ncats_dict[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth * global_prior) / (cnt + smooth)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth * global_prior) / (cnt_f + smooth)

            col_name = f"nte2_{col}"
            tr_te[col_name] = tr_vals
            va_te[col_name] = te_map_f[c_va].astype(np.float32)
            te_te[col_name] = te_map_f[c_te].astype(np.float32)

        X_tr_f = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te, index=X_dig2.index[tr_idx])], axis=1)
        X_va_f = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te, index=X_dig2.index[va_idx])], axis=1)
        X_te_f = pd.concat([X_test_dig2, pd.DataFrame(te_te, index=X_test_dig2.index)], axis=1)

        # Model A: 2-leaf decision stumps (pure additive GAM, zero interaction overfitting)
        m_stump = lgb.LGBMClassifier(
            n_estimators=8000, learning_rate=0.05, num_leaves=2, max_depth=1,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=fold, verbose=-1, n_jobs=-1,
        )
        m_stump.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_stump_oof[va_idx] = m_stump.predict_proba(X_va_f)[:, 1]
        lgb_stump_test += m_stump.predict_proba(X_te_f)[:, 1] / 5

        # Model B: 3-leaf shallow trees on upgraded features
        m_dig2 = lgb.LGBMClassifier(
            n_estimators=6000, learning_rate=0.04, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
            random_state=fold + 100, verbose=-1, n_jobs=-1,
        )
        m_dig2.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_dig2_oof[va_idx] = m_dig2.predict_proba(X_va_f)[:, 1]
        lgb_dig2_test += m_dig2.predict_proba(X_te_f)[:, 1] / 5

        print(f"Fold {fold} | 2-Leaf Stump: {roc_auc_score(y_va, lgb_stump_oof[va_idx]):.5f} (tree {m_stump.best_iteration_}) | 3-Leaf Dig2: {roc_auc_score(y_va, lgb_dig2_oof[va_idx]):.5f} (tree {m_dig2.best_iteration_})")

if "lgb_dig2_oof" not in globals():
    lgb_stump_oof = np.zeros(len(X_dig2))
    lgb_stump_test = np.zeros(len(X_test_dig2))
    lgb_dig2_oof = np.zeros(len(X_dig2))
    lgb_dig2_test = np.zeros(len(X_test_dig2))
print("X_dig2 ready:", X_dig2.shape, "| nte2_keys:", len(nte2_keys))

In [ ]:
# Step 10: Option 2 — Spline GLM + PyTorch GPU Residual MLP on X_dig2 + 47-Key Nested TE
import gc
import time
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def signed_log1p(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def safe_logit(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "glm_oof" not in globals() or "mlp_oof" not in globals():
    t0_all = time.time()
    n_tr = len(X_dig2)
    n_te = len(X_test_dig2)
    glm_oof = np.zeros(n_tr, dtype=np.float64)
    glm_test = np.zeros(n_te, dtype=np.float64)
    mlp_oof = np.zeros(n_tr, dtype=np.float64)
    mlp_test = np.zeros(n_te, dtype=np.float64)

    num_cols = [c for c in X_dig2.columns if c not in text_cols]
    log_skew_cols = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew_cols.add(c)

    # Pre-build signed-log1p numeric base matrices once (float32)
    base_tr = X_dig2[num_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base_te = X_test_dig2[num_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num_cols):
        if c in log_skew_cols:
            base_tr[:, j] = signed_log1p(base_tr[:, j])
            base_te[:, j] = signed_log1p(base_te[:, j])
    base_tr = np.nan_to_num(base_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base_te = np.nan_to_num(base_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline_base_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline_base_idx = [num_cols.index(c) for c in spline_base_cols]

    oh_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh_enc.fit_transform(X_dig2[oh_cols].astype(str))
    _oh_te = oh_enc.transform(X_test_dig2[oh_cols].astype(str))
    oh_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    emb_cols = [
        "merchant_category", "card_type", "auth_method", "channel", "device_type",
        "time_of_day_hour", "day_of_week", "cvv_retry_count", "prior_disputes",
        "amount_usd_cents", "hours_since_last_txn_cents", "account_balance_usd_cents",
        "velocity_score_cents", "merchant_risk_score_cents",
        "amount_usd_first_digit", "hours_since_last_txn_first_digit", "distance_from_home_km_first_digit"
    ]
    emb_ncats = [ncats_dict[c] for c in emb_cols]
    X_cat_all = np.column_stack([np.clip(codes_tr_dict[c], 0, ncats_dict[c] - 1) for c in emb_cols]).astype(np.int64)
    X_cat_te = np.column_stack([np.clip(codes_te_dict[c], 0, ncats_dict[c] - 1) for c in emb_cols]).astype(np.int64)

    top_nte_spline_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "account_balance_usd_cents", "velocity_score_cents", "merchant_risk_score_cents"
    ]
    top_nte_idx = [nte2_keys.index(k) for k in top_nte_spline_keys]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Option 2 running on device={device} | base_num={base_tr.shape[1]} | emb={len(emb_cols)} | oh={oh_all.shape[1]}")

    class ResBlock(nn.Module):
        def __init__(self, dim, dropout=0.25):
            super().__init__()
            self.block = nn.Sequential(
                nn.BatchNorm1d(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
                nn.BatchNorm1d(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
            )
        def forward(self, x):
            return x + self.block(x)

    class FraudResNet(nn.Module):
        def __init__(self, num_dim, card_sizes, hidden=256, dropout=0.25):
            super().__init__()
            self.embs = nn.ModuleList([
                nn.Embedding(nc + 1, min(16, max(4, (nc + 1) // 2))) for nc in card_sizes
            ])
            emb_dim = sum(e.embedding_dim for e in self.embs)
            in_dim = num_dim + emb_dim
            self.skip = nn.Linear(in_dim, 1)
            self.in_proj = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.BatchNorm1d(hidden),
                nn.SiLU(),
                nn.Dropout(0.15),
            )
            self.res1 = ResBlock(hidden, dropout)
            self.res2 = ResBlock(hidden, dropout)
            self.head = nn.Sequential(
                nn.BatchNorm1d(hidden),
                nn.SiLU(),
                nn.Linear(hidden, 1),
            )
        def forward(self, x_num, x_cat):
            e = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embs)], dim=1)
            x = torch.cat([x_num, e], dim=1)
            h = self.res2(self.res1(self.in_proj(x)))
            return (self.skip(x) + self.head(h)).squeeze(1)

    from sklearn.linear_model import LogisticRegression

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr_arr.astype(np.int32)))

        # Leak-free 47 logit(Nested TE) matrices for this fold
        nte_tr = np.empty((len(tr_idx), len(nte2_keys)), dtype=np.float32)
        nte_va = np.empty((len(va_idx), len(nte2_keys)), dtype=np.float32)
        nte_te = np.empty((n_te, len(nte2_keys)), dtype=np.float32)

        for j, col in enumerate(nte2_keys):
            c_all = codes_tr_dict[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes_te_dict[col]
            nc = ncats_dict[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth * global_prior) / (cnt + smooth)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth * global_prior) / (cnt_f + smooth)

            nte_tr[:, j] = safe_logit(tr_vals)
            nte_va[:, j] = safe_logit(te_map_f[c_va])
            nte_te[:, j] = safe_logit(te_map_f[c_te])

        # Standardize base_num + logit(NTE) inside the fold
        raw_tr = np.hstack([base_tr[tr_idx], nte_tr])
        raw_va = np.hstack([base_tr[va_idx], nte_va])
        raw_te = np.hstack([base_te, nte_te])

        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(raw_tr), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(raw_va), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(raw_te), -6.0, 6.0).astype(np.float32)

        # --- Model 1: Spline GLM (Logistic Regression with B-splines) ---
        t_glm = time.time()
        sp_cols_tr = np.hstack([x_tr_std[:, spline_base_idx], x_tr_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])
        sp_cols_va = np.hstack([x_va_std[:, spline_base_idx], x_va_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])
        sp_cols_te = np.hstack([x_te_std[:, spline_base_idx], x_te_std[:, base_tr.shape[1] + np.array(top_nte_idx)]])

        spline_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = spline_tf.fit_transform(np.clip(sp_cols_tr, -4.0, 4.0)).astype(np.float32)
        sp_va = spline_tf.transform(np.clip(sp_cols_va, -4.0, 4.0)).astype(np.float32)
        sp_te = spline_tf.transform(np.clip(sp_cols_te, -4.0, 4.0)).astype(np.float32)

        glm_tr = np.hstack([x_tr_std, sp_tr, oh_all[tr_idx]])
        glm_va = np.hstack([x_va_std, sp_va, oh_all[va_idx]])
        glm_te = np.hstack([x_te_std, sp_te, oh_te])

        if not globals().get("SKIP_SUPERSEDED", False):
            glm = LogisticRegression(C=0.2, solver="lbfgs", max_iter=250, tol=1e-4)
            glm.fit(glm_tr, y_tr_arr.astype(np.int32))
            glm_oof[va_idx] = glm.predict_proba(glm_va)[:, 1]
            glm_test += glm.predict_proba(glm_te)[:, 1] / 5.0
            glm_auc = roc_auc_score(y_va_arr, glm_oof[va_idx])
        else:
            glm_auc = 0.0
        glm_sec = time.time() - t_glm
        del sp_tr, sp_va, sp_te, glm_tr, glm_va, glm_te, sp_cols_tr, sp_cols_va, sp_cols_te, raw_tr, raw_va, raw_te
        gc.collect()

        # --- Model 2: PyTorch GPU Residual MLP ---
        t_mlp = time.time()
        torch.manual_seed(42 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42 + fold)

        X_tr_num_t = torch.from_numpy(x_tr_std).to(device)
        X_va_num_t = torch.from_numpy(x_va_std).to(device)
        X_te_num_t = torch.from_numpy(x_te_std).to(device)
        X_tr_cat_t = torch.from_numpy(X_cat_all[tr_idx]).to(device)
        X_va_cat_t = torch.from_numpy(X_cat_all[va_idx]).to(device)
        X_te_cat_t = torch.from_numpy(X_cat_te).to(device)
        y_tr_t = torch.from_numpy(y_tr_arr.astype(np.float32)).to(device)

        model = FraudResNet(x_tr_std.shape[1], emb_ncats, hidden=256, dropout=0.25).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-4)
        epochs = 22
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=2e-5)
        crit = nn.BCEWithLogitsLoss()
        bs = 2048
        n_f = len(tr_idx)

        best_auc = -1.0
        best_ep = 0
        best_state = None
        no_imp = 0

        for ep in range(1, epochs + 1):
            model.train()
            perm = torch.randperm(n_f, device=device)
            for s in range(0, n_f, bs):
                idx = perm[s:s + bs]
                opt.zero_grad(set_to_none=True)
                logits = model(X_tr_num_t[idx], X_tr_cat_t[idx])
                loss = crit(logits, y_tr_t[idx])
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            va_preds = []
            with torch.no_grad():
                for s in range(0, len(va_idx), 8192):
                    va_preds.append(torch.sigmoid(model(X_va_num_t[s:s + 8192], X_va_cat_t[s:s + 8192])).cpu().numpy())
            va_p = np.concatenate(va_preds)
            va_auc = roc_auc_score(y_va_arr, va_p)
            if va_auc > best_auc + 1e-6:
                best_auc = va_auc
                best_ep = ep
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
                if no_imp >= 5:
                    break

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            va_preds = [torch.sigmoid(model(X_va_num_t[s:s + 8192], X_va_cat_t[s:s + 8192])).cpu().numpy() for s in range(0, len(va_idx), 8192)]
            te_preds = [torch.sigmoid(model(X_te_num_t[s:s + 8192], X_te_cat_t[s:s + 8192])).cpu().numpy() for s in range(0, n_te, 8192)]
        mlp_oof[va_idx] = np.concatenate(va_preds)
        mlp_test += np.concatenate(te_preds) / 5.0
        mlp_sec = time.time() - t_mlp

        print(f"Fold {fold} | Spline GLM: {glm_auc:.5f} ({glm_sec:.1f}s) | PyTorch MLP: {best_auc:.5f} (ep {best_ep}, {mlp_sec:.1f}s)")

        del model, opt, sched, best_state, X_tr_num_t, X_va_num_t, X_te_num_t, X_tr_cat_t, X_va_cat_t, X_te_cat_t, y_tr_t
        del x_tr_std, x_va_std, x_te_std, nte_tr, nte_va, nte_te
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    np.savez("/kaggle/working/preds_opt2.npz", glm_oof=glm_oof, glm_test=glm_test, mlp_oof=mlp_oof, mlp_test=mlp_test)
    print(f"Total Option 2 runtime: {time.time() - t0_all:.1f}s")

print("\n--- Option 2 Models (5-fold OOF AUC) ---")
if glm_oof.max() > 0: print(f"Spline GLM  (X_dig2 + 47 logit-NTE): {roc_auc_score(y, glm_oof):.5f}")
print(f"PyTorch MLP (ResNet + Emb + NTE):    {roc_auc_score(y, mlp_oof):.5f}")


In [ ]:
# Step 11: Exp 3 — X_dig3 (Exact Tenths + Flags Bitmask + Group Z-Scores) + 28 High-Signal Pairwise Nested TE Keys
import gc
import time
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def _signed_log1p(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "X_dig3" not in globals():
    X_dig3 = X_dig2.copy()
    X_test_dig3 = X_test_dig2.copy()

    flag_cols = [
        "is_foreign_transaction", "is_new_merchant", "used_vpn",
        "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
    ]
    def _build_extras(df_tr, df_te):
        tr_out = pd.DataFrame(index=df_tr.index)
        te_out = pd.DataFrame(index=df_te.index)

        # Exact tenth integers & 6-bit flag pattern
        tr_out["risk_x10"] = np.round(df_tr["merchant_risk_score"].values * 10).astype(np.int32)
        te_out["risk_x10"] = np.round(df_te["merchant_risk_score"].values * 10).astype(np.int32)
        tr_out["vel_x10"] = np.round(df_tr["velocity_score"].values * 10).astype(np.int32)
        te_out["vel_x10"] = np.round(df_te["velocity_score"].values * 10).astype(np.int32)
        tr_out["hrs_x10"] = np.round(df_tr["hours_since_last_txn"].values * 10).astype(np.int32)
        te_out["hrs_x10"] = np.round(df_te["hours_since_last_txn"].values * 10).astype(np.int32)

        fc_tr = np.zeros(len(df_tr), dtype=np.int32)
        fc_te = np.zeros(len(df_te), dtype=np.int32)
        for bi, fc in enumerate(flag_cols):
            fc_tr += (df_tr[fc].values.astype(np.int32) << bi)
            fc_te += (df_te[fc].values.astype(np.int32) << bi)
        tr_out["flags_bitmask"] = fc_tr
        te_out["flags_bitmask"] = fc_te
        return tr_out, te_out

    ex_tr, ex_te = _build_extras(X_dig3, X_test_dig3)
    for c in ex_tr.columns:
        X_dig3[c] = ex_tr[c]
        X_test_dig3[c] = ex_te[c]

    # Within-group z-scores by merchant_category, auth_method, and flags_bitmask (feature-only X, zero label usage)
    comb_x = pd.concat([
        X_dig3[["merchant_category", "auth_method", "flags_bitmask", "amount_usd_log1p", "merchant_risk_score", "velocity_score", "hours_since_last_txn_log1p"]],
        X_test_dig3[["merchant_category", "auth_method", "flags_bitmask", "amount_usd_log1p", "merchant_risk_score", "velocity_score", "hours_since_last_txn_log1p"]]
    ], ignore_index=True)
    n_tr3 = len(X_dig3)
    for gcol in ["merchant_category", "auth_method", "flags_bitmask"]:
        for vcol in ["merchant_risk_score", "velocity_score", "amount_usd_log1p", "hours_since_last_txn_log1p"]:
            grp = comb_x.groupby(gcol, observed=False)[vcol]
            mu = grp.transform("mean").values
            sd = grp.transform("std").values + 1e-5
            z = ((comb_x[vcol].values - mu) / sd).astype(np.float32)
            cname = f"{vcol}_z_{gcol}"
            X_dig3[cname] = z[:n_tr3]
            X_test_dig3[cname] = z[n_tr3:]
    del comb_x, ex_tr, ex_te

if "codes3_tr" not in globals():
    _, amt_bins = pd.qcut(X_dig3["amount_usd"], 20, retbins=True, labels=False, duplicates="drop")
    amt_bin_tr = np.digitize(X_dig3["amount_usd"].values, amt_bins[1:-1])
    amt_bin_te = np.digitize(X_test_dig3["amount_usd"].values, amt_bins[1:-1])

    def _pair_series(df, amt_b):
        age10 = (df["customer_age"].values // 10).astype(str)
        s_flags = df["flags_bitmask"].astype(str)
        s_rcnt = df["risk_flag_count"].astype(str)
        s_rtens = df["merchant_risk_score_tens"].astype(str)
        s_vtens = df["velocity_score_tens"].astype(str)
        s_auth = df["auth_method"].astype(str)
        s_merch = df["merchant_category"].astype(str)
        s_cvv = df["cvv_retry_count"].astype(str)
        return {
            "risk_x10": df["risk_x10"].astype(str),
            "vel_x10": df["vel_x10"].astype(str),
            "hrs_x10": df["hrs_x10"].astype(str),
            "flags_bitmask": s_flags,
            "flags_x_riskTens": s_flags + "_" + s_rtens,
            "riskCount_x_riskTens": s_rcnt + "_" + s_rtens,
            "riskCount_x_cvv": s_rcnt + "_" + s_cvv,
            "riskCount_x_velTens": s_rcnt + "_" + s_vtens,
            "flags_x_auth": s_flags + "_" + s_auth,
            "risk_x_auth": s_rcnt + "_" + s_auth,
            "auth_x_cvv": s_auth + "_" + s_cvv,
            "flags_x_merch": s_flags + "_" + s_merch,
            "riskTens_x_auth": s_rtens + "_" + s_auth,
            "riskTens_x_velTens": s_rtens + "_" + s_vtens,
            "merch_x_cvv": s_merch + "_" + s_cvv,
            "velTens_x_auth": s_vtens + "_" + s_auth,
            "riskTens_x_age10": s_rtens + "_" + pd.Series(age10, index=df.index),
            "riskTens_x_merch": s_rtens + "_" + s_merch,
            "merch_x_auth": s_merch + "_" + s_auth,
            "cvv_x_disp": s_cvv + "_" + df["prior_disputes"].astype(str),
            "amtLogBin_x_riskTens": pd.Series(amt_b.astype(str), index=df.index) + "_" + s_rtens,
            "merch_x_new": s_merch + "_" + df["is_new_merchant"].astype(str),
            "hrsInt_x_velTens": df["hours_since_last_txn_int"].astype(str) + "_" + s_vtens,
            "txn24_x_velTens": df["txn_count_last_24h"].astype(str) + "_" + s_vtens,
            "auth_x_scam": s_auth + "_" + df["is_ai_generated_scam_attempt"].astype(str),
            "auth_x_dev": s_auth + "_" + df["device_type"].astype(str),
            "vpn_x_ipmm": df["used_vpn"].astype(str) + "_" + df["ip_country_mismatch"].astype(str),
            "foreign_x_ipmm": df["is_foreign_transaction"].astype(str) + "_" + df["ip_country_mismatch"].astype(str),
        }

    p_tr_dict = _pair_series(X_dig3, amt_bin_tr)
    p_te_dict = _pair_series(X_test_dig3, amt_bin_te)
    pair_keys = list(p_tr_dict.keys())
    n_tr3 = len(X_dig3)

    codes3_tr = dict(codes_tr_dict)
    codes3_te = dict(codes_te_dict)
    ncats3 = dict(ncats_dict)
    for col in pair_keys:
        comb = pd.concat([p_tr_dict[col], p_te_dict[col]], ignore_index=True)
        codes, _ = pd.factorize(comb, sort=False)
        codes3_tr[col] = codes[:n_tr3]
        codes3_te[col] = codes[n_tr3:]
        ncats3[col] = int(codes.max()) + 1

    # Add frequency encoding for high-signal interaction keys
    for col in ["flags_bitmask", "flags_x_riskTens", "risk_x_auth", "riskTens_x_velTens"]:
        comb_c = np.concatenate([codes3_tr[col], codes3_te[col]])
        vc = np.bincount(comb_c, minlength=ncats3[col]).astype(np.float32)
        X_dig3[f"{col}_freq"] = vc[codes3_tr[col]]
        X_test_dig3[f"{col}_freq"] = vc[codes3_te[col]]

    nte3_keys = list(nte2_keys) + [k for k in pair_keys if k not in nte2_keys]
    del p_tr_dict, p_te_dict
    print(f"Prepared X_dig3: {X_dig3.shape} | Total Nested TE keys: {len(nte3_keys)}")

if "lgb_dig3_oof" not in globals() or "xgb_dig3_oof" not in globals() or "cb_dig3_oof" not in globals() or "glm_dig3_oof" not in globals():
    t0_exp3 = time.time()
    n_tr3 = len(X_dig3)
    n_te3 = len(X_test_dig3)
    lgb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    lgb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    xgb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    xgb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    cb_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    cb_dig3_test = np.zeros(n_te3, dtype=np.float64)
    glm_dig3_oof = np.zeros(n_tr3, dtype=np.float64)
    glm_dig3_test = np.zeros(n_te3, dtype=np.float64)

    # Prebuild base numeric + one-hot arrays for Spline_GLM_Dig3
    num3_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew3 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num3_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew3.add(c)
    base3_tr = X_dig3[num3_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base3_te = X_test_dig3[num3_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num3_cols):
        if c in log_skew3:
            base3_tr[:, j] = _signed_log1p(base3_tr[:, j])
            base3_te[:, j] = _signed_log1p(base3_te[:, j])
    base3_tr = np.nan_to_num(base3_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base3_te = np.nan_to_num(base3_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline3_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline3_idx = [num3_cols.index(c) for c in spline3_cols]
    top_nte3_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "flags_x_riskTens", "riskCount_x_cvv", "risk_x_auth", "riskTens_x_velTens", "flags_bitmask"
    ]
    top_nte3_idx = [nte3_keys.index(k) for k in top_nte3_keys]

    oh3_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh3_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh3_enc.fit_transform(X_dig3[oh3_cols].astype(str))
    _oh_te = oh3_enc.transform(X_test_dig3[oh3_cols].astype(str))
    oh3_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh3_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    smooth3 = 25.0
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig3, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        tr_te, va_te, te_te = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te3, len(nte3_keys)), dtype=np.float32)

        for j, col in enumerate(nte3_keys):
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + smooth3 * global_prior) / (cnt + smooth3)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + smooth3 * global_prior) / (cnt_f + smooth3)

            cname = f"nte3_{col}"
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)
            tr_te[cname] = tr_vals
            va_te[cname] = va_vals
            te_te[cname] = te_vals

            nte_log_tr[:, j] = _safe_logit(tr_vals)
            nte_log_va[:, j] = _safe_logit(va_vals)
            nte_log_te[:, j] = _safe_logit(te_vals)

        X_tr_f = pd.concat([X_dig3.iloc[tr_idx], pd.DataFrame(tr_te, index=X_dig3.index[tr_idx])], axis=1)
        X_va_f = pd.concat([X_dig3.iloc[va_idx], pd.DataFrame(va_te, index=X_dig3.index[va_idx])], axis=1)
        X_te_f = pd.concat([X_test_dig3, pd.DataFrame(te_te, index=X_test_dig3.index)], axis=1)

        # 1) Shallow LightGBM (3 leaves) on X_dig3
        m_lgb = lgb.LGBMClassifier(
            n_estimators=7000, learning_rate=0.035, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.65,
            reg_alpha=1.0, reg_lambda=5.0, random_state=fold + 300, verbose=-1, n_jobs=-1,
        )
        m_lgb.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_dig3_oof[va_idx] = m_lgb.predict_proba(X_va_f)[:, 1]
        lgb_dig3_test += m_lgb.predict_proba(X_te_f)[:, 1] / 5.0

        # 2) Shallow XGBoost (max_depth=2, GPU) on X_dig3
        m_xgb = xgb.XGBClassifier(
            n_estimators=6000, learning_rate=0.04, max_depth=2,
            min_child_weight=100, subsample=0.8, colsample_bytree=0.7,
            reg_alpha=1.0, reg_lambda=5.0,
            tree_method="hist", device="cuda", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=150, random_state=fold + 300, n_jobs=-1,
        )
        m_xgb.fit(X_tr_f, y_tr, eval_set=[(X_va_f, y_va)], verbose=False)
        xgb_dig3_oof[va_idx] = m_xgb.predict_proba(X_va_f)[:, 1]
        xgb_dig3_test += m_xgb.predict_proba(X_te_f)[:, 1] / 5.0

        # 3) Shallow CatBoost (depth=3, GPU) on X_dig3
        X_tr_cb, X_va_cb, X_te_cb = X_tr_f.copy(), X_va_f.copy(), X_te_f.copy()
        for c in text_cols:
            X_tr_cb[c] = X_tr_cb[c].cat.codes
            X_va_cb[c] = X_va_cb[c].cat.codes
            X_te_cb[c] = X_te_cb[c].cat.codes
        m_cb = CatBoostClassifier(
            iterations=6000, learning_rate=0.045, depth=3,
            l2_leaf_reg=5.0, cat_features=text_cols, one_hot_max_size=25,
            eval_metric="AUC", random_seed=fold + 300, task_type="GPU", verbose=False,
        )
        m_cb.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), early_stopping_rounds=150, verbose=False)
        cb_dig3_oof[va_idx] = m_cb.predict_proba(X_va_cb)[:, 1]
        cb_dig3_test += m_cb.predict_proba(X_te_cb)[:, 1] / 5.0
        del X_tr_f, X_va_f, X_te_f, X_tr_cb, X_va_cb, X_te_cb, tr_te, va_te, te_te

        # 4) Spline GLM on X_dig3 + 75 logit(NTE)
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base3_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base3_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base3_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        off = base3_tr.shape[1]
        sp_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = sp_tf.fit_transform(np.clip(np.hstack([x_tr_std[:, spline3_idx], x_tr_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_va = sp_tf.transform(np.clip(np.hstack([x_va_std[:, spline3_idx], x_va_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_te = sp_tf.transform(np.clip(np.hstack([x_te_std[:, spline3_idx], x_te_std[:, off + np.array(top_nte3_idx)]]), -4.0, 4.0)).astype(np.float32)

        if not globals().get("SKIP_SUPERSEDED", False):
            glm3 = LogisticRegression(C=0.2, solver="lbfgs", max_iter=250, tol=1e-4)
            glm3.fit(np.hstack([x_tr_std, sp_tr, oh3_all[tr_idx]]), y_tr_arr.astype(np.int32))
            glm_dig3_oof[va_idx] = glm3.predict_proba(np.hstack([x_va_std, sp_va, oh3_all[va_idx]]))[:, 1]
            glm_dig3_test += glm3.predict_proba(np.hstack([x_te_std, sp_te, oh3_te]))[:, 1] / 5.0
        else:
            glm_dig3_oof[va_idx] = 0.5 * lgb_dig3_oof[va_idx] + 0.5 * xgb_dig3_oof[va_idx]

        print(
            f"Fold {fold} ({time.time()-t_f:.0f}s) | LGB3: {roc_auc_score(y_va, lgb_dig3_oof[va_idx]):.5f} "
            f"| XGB3: {roc_auc_score(y_va, xgb_dig3_oof[va_idx]):.5f} "
            f"| CB3: {roc_auc_score(y_va, cb_dig3_oof[va_idx]):.5f} "
            f"| GLM3: {roc_auc_score(y_va, glm_dig3_oof[va_idx]):.5f}"
        )
        del x_tr_std, x_va_std, x_te_std, sp_tr, sp_va, sp_te, nte_log_tr, nte_log_va, nte_log_te
        gc.collect()

    np.savez(
        "/kaggle/working/preds_exp3.npz",
        lgb_dig3_oof=lgb_dig3_oof, lgb_dig3_test=lgb_dig3_test,
        xgb_dig3_oof=xgb_dig3_oof, xgb_dig3_test=xgb_dig3_test,
        cb_dig3_oof=cb_dig3_oof, cb_dig3_test=cb_dig3_test,
        glm_dig3_oof=glm_dig3_oof, glm_dig3_test=glm_dig3_test,
    )
    print(f"Total Exp 3 runtime: {time.time() - t0_exp3:.1f}s")

print("\n--- Exp 3 (X_dig3 + Pairwise Nested TE) 5-fold OOF AUCs ---")
print(f"LGB_3L_Dig3:     {roc_auc_score(y, lgb_dig3_oof):.5f}")
print(f"XGB_Dig3:        {roc_auc_score(y, xgb_dig3_oof):.5f}")
print(f"CB_Dig3:         {roc_auc_score(y, cb_dig3_oof):.5f}")
print(f"Spline_GLM_Dig3: {roc_auc_score(y, glm_dig3_oof):.5f}")


In [ ]:
# Step 12: Exp 4 — PyTorch Periodic-Embedding WideResNet (Dig3) + Clean-Pair LightGBM (3-Leaf)
import gc
import math
import time
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def _signed_log1p_4(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit_4(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "nn_dig3_oof" not in globals() or "lgb_clean_oof" not in globals():
    t0_exp4 = time.time()
    n_tr4 = len(X_dig3)
    n_te4 = len(X_test_dig3)
    nn_dig3_oof = np.zeros(n_tr4, dtype=np.float64)
    nn_dig3_test = np.zeros(n_te4, dtype=np.float64)
    lgb_clean_oof = np.zeros(n_tr4, dtype=np.float64)
    lgb_clean_test = np.zeros(n_te4, dtype=np.float64)

    # Numeric base matrix for PyTorch PLR WideResNet
    num4_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew4 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num4_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew4.add(c)
    base4_tr = X_dig3[num4_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base4_te = X_test_dig3[num4_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num4_cols):
        if c in log_skew4:
            base4_tr[:, j] = _signed_log1p_4(base4_tr[:, j])
            base4_te[:, j] = _signed_log1p_4(base4_te[:, j])
    base4_tr = np.nan_to_num(base4_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base4_te = np.nan_to_num(base4_te, nan=0.0, posinf=0.0, neginf=0.0)

    oh4_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh4_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh4_enc.fit_transform(X_dig3[oh4_cols].astype(str))
    _oh_te = oh4_enc.transform(X_test_dig3[oh4_cols].astype(str))
    oh4_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh4_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    # Clean 17 pairwise / exact-tenth keys (avoiding 437-bucket flags_x_merch & raw flags_bitmask split clutter)
    clean_extra_keys = [
        "hrs_x10", "risk_x10", "vel_x10", "flags_bitmask", "flags_x_riskTens",
        "riskCount_x_riskTens", "riskCount_x_cvv", "riskCount_x_velTens",
        "flags_x_auth", "risk_x_auth", "auth_x_cvv", "riskTens_x_auth",
        "riskTens_x_velTens", "merch_x_cvv", "velTens_x_auth", "merch_x_auth", "cvv_x_disp"
    ]
    lgb_nte_keys = list(nte2_keys) + clean_extra_keys

    class PeriodicEmbeddings(nn.Module):
        def __init__(self, n_features, n_bins=6, d_emb=4, sigma=0.12):
            super().__init__()
            self.freqs = nn.Parameter(torch.randn(n_features, n_bins) * sigma)
            self.weight = nn.Parameter(torch.randn(n_features, 2 * n_bins, d_emb) / math.sqrt(2 * n_bins))
            self.bias = nn.Parameter(torch.zeros(n_features, d_emb))
        def forward(self, x):
            c = 2.0 * math.pi * x.unsqueeze(-1) * self.freqs.unsqueeze(0)
            sc = torch.cat([torch.sin(c), torch.cos(c)], dim=-1)
            emb = torch.einsum("bdk,dkm->bdm", sc, self.weight) + self.bias.unsqueeze(0)
            return torch.nn.functional.silu(emb).flatten(1)

    class PLRWideResNet(nn.Module):
        def __init__(self, n_num, n_oh, n_bins=6, d_emb=4, hidden=192, dropout=0.35):
            super().__init__()
            self.plr = PeriodicEmbeddings(n_num, n_bins=n_bins, d_emb=d_emb, sigma=0.12)
            raw_dim = n_num + n_oh
            in_dim = n_num * d_emb + raw_dim
            self.wide = nn.Linear(raw_dim, 1)
            self.in_proj = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.LayerNorm(hidden),
                nn.SiLU(),
                nn.Dropout(dropout),
            )
            self.b1 = nn.Sequential(
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            )
            self.b2 = nn.Sequential(
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            )
            self.act = nn.SiLU()
            self.head = nn.Linear(hidden, 1)
        def forward(self, x_num, x_oh):
            raw = torch.cat([x_num, x_oh], dim=1)
            p = self.plr(x_num)
            h = self.in_proj(torch.cat([p, raw], dim=1))
            h = self.act(h + self.b1(h))
            h = self.act(h + self.b2(h))
            return (self.wide(raw) + self.head(h)).squeeze(1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Exp 4 running on device={device} | PLR n_num={base4_tr.shape[1]+len(nte3_keys)} | LGB_Clean NTE keys={len(lgb_nte_keys)}")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dig2, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        y_va_arr = y_arr[va_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        # Compute all 75 NTEs cleanly (smooth=20 for clean LGB compatibility, smooth=25 for NN)
        tr_te_clean, va_te_clean, te_te_clean = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te4, len(nte3_keys)), dtype=np.float32)
        clean_set = set(lgb_nte_keys)

        for j, col in enumerate(nte3_keys):
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + 20.0 * global_prior) / (cnt + 20.0)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + 20.0 * global_prior) / (cnt_f + 20.0)
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)

            if col in clean_set:
                cname = f"nte_{col}"
                tr_te_clean[cname] = tr_vals
                va_te_clean[cname] = va_vals
                te_te_clean[cname] = te_vals

            nte_log_tr[:, j] = _safe_logit_4(tr_vals)
            nte_log_va[:, j] = _safe_logit_4(va_vals)
            nte_log_te[:, j] = _safe_logit_4(te_vals)

        # 1) PyTorch PLR WideResNet on GPU
        t_nn = time.time()
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base4_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base4_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base4_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        torch.manual_seed(100 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(100 + fold)

        X_tr_n = torch.from_numpy(x_tr_std).to(device)
        X_va_n = torch.from_numpy(x_va_std).to(device)
        X_te_n = torch.from_numpy(x_te_std).to(device)
        X_tr_o = torch.from_numpy(oh4_all[tr_idx]).to(device)
        X_va_o = torch.from_numpy(oh4_all[va_idx]).to(device)
        X_te_o = torch.from_numpy(oh4_te).to(device)
        y_tr_t = torch.from_numpy(y_tr_arr.astype(np.float32)).to(device)

        model = PLRWideResNet(x_tr_std.shape[1], oh4_all.shape[1], n_bins=6, d_emb=4, hidden=192, dropout=0.35).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=2e-3)
        epochs = 26
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
        crit = nn.BCEWithLogitsLoss()
        bs = 4096
        n_f = len(tr_idx)

        best_auc, best_ep, best_state, no_imp = -1.0, 0, None, 0
        for ep in range(1, epochs + 1):
            model.train()
            perm = torch.randperm(n_f, device=device)
            for s in range(0, n_f, bs):
                idx = perm[s:s + bs]
                opt.zero_grad(set_to_none=True)
                loss = crit(model(X_tr_n[idx], X_tr_o[idx]), y_tr_t[idx])
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            with torch.no_grad():
                va_p = np.concatenate([
                    torch.sigmoid(model(X_va_n[s:s + 8192], X_va_o[s:s + 8192])).cpu().numpy()
                    for s in range(0, len(va_idx), 8192)
                ])
            va_auc = roc_auc_score(y_va_arr, va_p)
            if va_auc > best_auc + 1e-6:
                best_auc, best_ep = va_auc, ep
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
                if no_imp >= 6:
                    break

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            nn_dig3_oof[va_idx] = np.concatenate([
                torch.sigmoid(model(X_va_n[s:s + 8192], X_va_o[s:s + 8192])).cpu().numpy()
                for s in range(0, len(va_idx), 8192)
            ])
            nn_dig3_test += np.concatenate([
                torch.sigmoid(model(X_te_n[s:s + 8192], X_te_o[s:s + 8192])).cpu().numpy()
                for s in range(0, n_te4, 8192)
            ]) / 5.0
        nn_sec = time.time() - t_nn

        del model, opt, sched, best_state, X_tr_n, X_va_n, X_te_n, X_tr_o, X_va_o, X_te_o, y_tr_t
        del x_tr_std, x_va_std, x_te_std, nte_log_tr, nte_log_va, nte_log_te
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # 2) Clean-Pair LightGBM (3-leaf on X_dig2 + 64 clean NTE keys)
        X_tr_c = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te_clean, index=X_dig2.index[tr_idx])], axis=1)
        X_va_c = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te_clean, index=X_dig2.index[va_idx])], axis=1)
        X_te_c = pd.concat([X_test_dig2, pd.DataFrame(te_te_clean, index=X_test_dig2.index)], axis=1)

        m_clean = lgb.LGBMClassifier(
            n_estimators=6500, learning_rate=0.04, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.75,
            random_state=fold + 400, verbose=-1, n_jobs=-1,
        )
        m_clean.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        lgb_clean_oof[va_idx] = m_clean.predict_proba(X_va_c)[:, 1]
        lgb_clean_test += m_clean.predict_proba(X_te_c)[:, 1] / 5.0
        c_auc = roc_auc_score(y_va, lgb_clean_oof[va_idx])

        print(f"Fold {fold} ({time.time()-t_f:.0f}s) | PLR_NN: {best_auc:.5f} (ep {best_ep}, {nn_sec:.1f}s) | LGB_CleanPairs: {c_auc:.5f} (tree {m_clean.best_iteration_})")
        del X_tr_c, X_va_c, X_te_c, tr_te_clean, va_te_clean, te_te_clean
        gc.collect()

    np.savez("/kaggle/working/preds_exp4.npz", nn_dig3_oof=nn_dig3_oof, nn_dig3_test=nn_dig3_test, lgb_clean_oof=lgb_clean_oof, lgb_clean_test=lgb_clean_test)
    print(f"Total Exp 4 runtime: {time.time() - t0_exp4:.1f}s")

print("\n--- Exp 4 Models (5-fold OOF AUC) ---")
print(f"PyTorch PLR WideResNet: {roc_auc_score(y, nn_dig3_oof):.5f}  (prev PyTorch MLP: {roc_auc_score(y, mlp_oof):.5f})")
print(f"LGB_CleanPairs (3L):    {roc_auc_score(y, lgb_clean_oof):.5f}")


In [ ]:
# Step 13: Exp 5 (Option B) — 10-Fold Upgrade + Randomized extra_trees=True LightGBM
import gc
import time
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder

def _signed_log1p_5(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def _safe_logit_5(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

if "lgb_dig2_10f_oof" not in globals() or "lgb_et_10f_oof" not in globals() or "glm_dig3_10f_oof" not in globals():
    t0_exp5 = time.time()
    n_tr5 = len(X_dig2)
    n_te5 = len(X_test_dig2)
    lgb_dig2_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    lgb_dig2_10f_test = np.zeros(n_te5, dtype=np.float64)
    lgb_et_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    lgb_et_10f_test = np.zeros(n_te5, dtype=np.float64)
    glm_dig3_10f_oof = np.zeros(n_tr5, dtype=np.float64)
    glm_dig3_10f_test = np.zeros(n_te5, dtype=np.float64)

    # Prebuild Spline GLM Dig3 base arrays once
    num5_cols = [c for c in X_dig3.columns if c not in text_cols]
    log_skew5 = set([
        "amount_usd", "hours_since_last_txn", "distance_from_home_km",
        "account_balance_usd", "amount_to_balance", "balance_after_txn",
        "txn_burst_rate", "amount_x_merchant_risk", "txn_count_last_24h"
    ])
    for c in num5_cols:
        if c.endswith(("_int", "_div10", "_div100", "_freq")):
            log_skew5.add(c)
    base5_tr = X_dig3[num5_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    base5_te = X_test_dig3[num5_cols].to_numpy(dtype=np.float32, na_value=0.0).copy()
    for j, c in enumerate(num5_cols):
        if c in log_skew5:
            base5_tr[:, j] = _signed_log1p_5(base5_tr[:, j])
            base5_te[:, j] = _signed_log1p_5(base5_te[:, j])
    base5_tr = np.nan_to_num(base5_tr, nan=0.0, posinf=0.0, neginf=0.0)
    base5_te = np.nan_to_num(base5_te, nan=0.0, posinf=0.0, neginf=0.0)

    spline5_cols = [
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "distance_from_home_km_log1p", "account_balance_usd_log1p",
        "velocity_score", "merchant_risk_score",
        "customer_age", "card_age_months",
        "amount_to_balance", "balance_after_txn", "txn_burst_rate", "amount_x_merchant_risk"
    ]
    spline5_idx = [num5_cols.index(c) for c in spline5_cols]
    top_nte5_keys = [
        "merchant_category", "amount_usd_cents", "hours_since_last_txn_cents",
        "flags_x_riskTens", "riskCount_x_cvv", "risk_x_auth", "riskTens_x_velTens", "flags_bitmask"
    ]
    top_nte5_idx = [nte3_keys.index(k) for k in top_nte5_keys]

    oh5_cols = list(text_cols) + ["time_of_day_hour", "day_of_week"]
    oh5_enc = OneHotEncoder(handle_unknown="ignore")
    _oh_tr = oh5_enc.fit_transform(X_dig3[oh5_cols].astype(str))
    _oh_te = oh5_enc.transform(X_test_dig3[oh5_cols].astype(str))
    oh5_all = (_oh_tr.toarray() if sparse.issparse(_oh_tr) else _oh_tr).astype(np.float32)
    oh5_te = (_oh_te.toarray() if sparse.issparse(_oh_te) else _oh_te).astype(np.float32)
    del _oh_tr, _oh_te

    skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=77)
    nte2_set = set(nte2_keys)
    print(f"Exp 5 (10-Fold + extra_trees) starting | X_dig2={X_dig2.shape} | X_dig3={X_dig3.shape}")

    for fold, (tr_idx, va_idx) in enumerate(skf10.split(X_dig2, y)):
        t_f = time.time()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        y_tr_arr = y_arr[tr_idx]
        global_prior = y_tr_arr.mean()
        inner_splits = list(inner_skf.split(tr_idx, y_tr))

        tr_te2, va_te2, te_te2 = {}, {}, {}
        tr_te3, va_te3, te_te3 = {}, {}, {}
        nte_log_tr = np.empty((len(tr_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_va = np.empty((len(va_idx), len(nte3_keys)), dtype=np.float32)
        nte_log_te = np.empty((n_te5, len(nte3_keys)), dtype=np.float32)

        for j, col in enumerate(nte3_keys):
            sm_val = 20.0 if col in nte2_set else 25.0
            c_all = codes3_tr[col]
            c_tr = c_all[tr_idx]
            c_va = c_all[va_idx]
            c_te = codes3_te[col]
            nc = ncats3[col]

            tr_vals = np.full(len(tr_idx), global_prior, dtype=np.float32)
            for in_tr, in_va in inner_splits:
                cnt = np.bincount(c_tr[in_tr], minlength=nc)
                sm = np.bincount(c_tr[in_tr], weights=y_tr_arr[in_tr], minlength=nc)
                te_map = (sm + sm_val * global_prior) / (cnt + sm_val)
                tr_vals[in_va] = te_map[c_tr[in_va]]

            cnt_f = np.bincount(c_tr, minlength=nc)
            sm_f = np.bincount(c_tr, weights=y_tr_arr, minlength=nc)
            te_map_f = (sm_f + sm_val * global_prior) / (cnt_f + sm_val)
            va_vals = te_map_f[c_va].astype(np.float32)
            te_vals = te_map_f[c_te].astype(np.float32)

            if col in nte2_set:
                k2 = f"nte2_{col}"
                tr_te2[k2] = tr_vals
                va_te2[k2] = va_vals
                te_te2[k2] = te_vals
            k3 = f"nte3_{col}"
            tr_te3[k3] = tr_vals
            va_te3[k3] = va_vals
            te_te3[k3] = te_vals

            nte_log_tr[:, j] = _safe_logit_5(tr_vals)
            nte_log_va[:, j] = _safe_logit_5(va_vals)
            nte_log_te[:, j] = _safe_logit_5(te_vals)

        # 1) 10-Fold Shallow 3-Leaf LGB on X_dig2 + 47 NTE keys
        X_tr_2 = pd.concat([X_dig2.iloc[tr_idx], pd.DataFrame(tr_te2, index=X_dig2.index[tr_idx])], axis=1)
        X_va_2 = pd.concat([X_dig2.iloc[va_idx], pd.DataFrame(va_te2, index=X_dig2.index[va_idx])], axis=1)
        X_te_2 = pd.concat([X_test_dig2, pd.DataFrame(te_te2, index=X_test_dig2.index)], axis=1)

        m_dig2_10 = lgb.LGBMClassifier(
            n_estimators=7000, learning_rate=0.045, num_leaves=3, max_depth=2,
            min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
            random_state=fold + 1000, verbose=-1, n_jobs=-1,
        )
        m_dig2_10.fit(X_tr_2, y_tr, eval_set=[(X_va_2, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_dig2_10f_oof[va_idx] = m_dig2_10.predict_proba(X_va_2)[:, 1]
        lgb_dig2_10f_test += m_dig2_10.predict_proba(X_te_2)[:, 1] / 10.0
        del X_tr_2, X_va_2, X_te_2, tr_te2, va_te2, te_te2

        # 2) 10-Fold Randomized extra_trees=True LightGBM (num_leaves=4, max_depth=3) on X_dig3 + 75 NTE keys
        X_tr_3 = pd.concat([X_dig3.iloc[tr_idx], pd.DataFrame(tr_te3, index=X_dig3.index[tr_idx])], axis=1)
        X_va_3 = pd.concat([X_dig3.iloc[va_idx], pd.DataFrame(va_te3, index=X_dig3.index[va_idx])], axis=1)
        X_te_3 = pd.concat([X_test_dig3, pd.DataFrame(te_te3, index=X_test_dig3.index)], axis=1)

        m_et_10 = lgb.LGBMClassifier(
            n_estimators=9000, learning_rate=0.045, num_leaves=4, max_depth=3,
            extra_trees=True, min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
            random_state=fold + 2000, verbose=-1, n_jobs=-1,
        )
        m_et_10.fit(X_tr_3, y_tr, eval_set=[(X_va_3, y_va)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
        lgb_et_10f_oof[va_idx] = m_et_10.predict_proba(X_va_3)[:, 1]
        lgb_et_10f_test += m_et_10.predict_proba(X_te_3)[:, 1] / 10.0
        del X_tr_3, X_va_3, X_te_3, tr_te3, va_te3, te_te3

        # 3) 10-Fold Spline GLM on X_dig3 + 75 logit(NTE)
        scaler = StandardScaler()
        x_tr_std = np.clip(scaler.fit_transform(np.hstack([base5_tr[tr_idx], nte_log_tr])), -6.0, 6.0).astype(np.float32)
        x_va_std = np.clip(scaler.transform(np.hstack([base5_tr[va_idx], nte_log_va])), -6.0, 6.0).astype(np.float32)
        x_te_std = np.clip(scaler.transform(np.hstack([base5_te, nte_log_te])), -6.0, 6.0).astype(np.float32)

        off = base5_tr.shape[1]
        sp_tf = SplineTransformer(n_knots=7, degree=3, knots="uniform", include_bias=False)
        sp_tr = sp_tf.fit_transform(np.clip(np.hstack([x_tr_std[:, spline5_idx], x_tr_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_va = sp_tf.transform(np.clip(np.hstack([x_va_std[:, spline5_idx], x_va_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)
        sp_te = sp_tf.transform(np.clip(np.hstack([x_te_std[:, spline5_idx], x_te_std[:, off + np.array(top_nte5_idx)]]), -4.0, 4.0)).astype(np.float32)

        glm5 = LogisticRegression(C=0.2, solver="lbfgs", max_iter=200, tol=1e-4)
        glm5.fit(np.hstack([x_tr_std, sp_tr, oh5_all[tr_idx]]), y_tr_arr.astype(np.int32))
        glm_dig3_10f_oof[va_idx] = glm5.predict_proba(np.hstack([x_va_std, sp_va, oh5_all[va_idx]]))[:, 1]
        glm_dig3_10f_test += glm5.predict_proba(np.hstack([x_te_std, sp_te, oh5_te]))[:, 1] / 10.0

        print(
            f"Fold {fold:2d}/10 ({time.time()-t_f:.0f}s) | LGB_3L_10F: {roc_auc_score(y_va, lgb_dig2_10f_oof[va_idx]):.5f} (t={m_dig2_10.best_iteration_}) "
            f"| ET_10F: {roc_auc_score(y_va, lgb_et_10f_oof[va_idx]):.5f} (t={m_et_10.best_iteration_}) "
            f"| GLM3_10F: {roc_auc_score(y_va, glm_dig3_10f_oof[va_idx]):.5f}"
        )
        del x_tr_std, x_va_std, x_te_std, sp_tr, sp_va, sp_te, nte_log_tr, nte_log_va, nte_log_te
        gc.collect()

    np.savez(
        "/kaggle/working/preds_exp5.npz",
        lgb_dig2_10f_oof=lgb_dig2_10f_oof, lgb_dig2_10f_test=lgb_dig2_10f_test,
        lgb_et_10f_oof=lgb_et_10f_oof, lgb_et_10f_test=lgb_et_10f_test,
        glm_dig3_10f_oof=glm_dig3_10f_oof, glm_dig3_10f_test=glm_dig3_10f_test,
    )
    print(f"Total Exp 5 runtime: {time.time() - t0_exp5:.1f}s")

print("\n--- Exp 5 (10-Fold + extra_trees) Overall OOF AUCs ---")
print(f"LGB_3L_Dig2_10F:  {roc_auc_score(y, lgb_dig2_10f_oof):.5f}")
print(f"LGB_ET_Dig3_10F:  {roc_auc_score(y, lgb_et_10f_oof):.5f}")
print(f"Spline_GLM3_10F:  {roc_auc_score(y, glm_dig3_10f_oof):.5f}")


In [ ]:
# Step 14: Option A — Context-Aware 2nd-Level Meta-Stacker (Dynamic Regime Weights + Tail Splines + GBDT Stacker)
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.model_selection import StratifiedKFold

cand_models = [
    ("LGB_3L_Dig2_10F", lgb_dig2_10f_oof, lgb_dig2_10f_test),
    ("LGB_ET_Dig3_10F", lgb_et_10f_oof, lgb_et_10f_test),
    ("Spline_GLM3_10F", glm_dig3_10f_oof, glm_dig3_10f_test),
    ("LGB_CleanPairs",  lgb_clean_oof,    lgb_clean_test),
    ("PLR_WideResNet",  nn_dig3_oof,      nn_dig3_test),
    ("LGB_3L_Dig3",     lgb_dig3_oof,     lgb_dig3_test),
    ("XGB_Dig3",        xgb_dig3_oof,     xgb_dig3_test),
    ("CB_Dig3",         cb_dig3_oof,      cb_dig3_test),
    ("Shallow_LGB_Dig", lgb_shallow_oof,  lgb_shallow_test),
    ("PyTorch_MLP",     mlp_oof,          mlp_test),
]
for _nm, _oof, _te in [
    ("Shallow_3L_Dig2", lgb_dig2_oof, lgb_dig2_test),
    ("Spline_GLM_Dig3", glm_dig3_oof, glm_dig3_test),
]:
    if _oof.max() > 0 and not np.allclose(_oof, 0.5 * lgb_dig3_oof + 0.5 * xgb_dig3_oof):
        cand_models.append((_nm, _oof, _te))

names = [m[0] for m in cand_models]
oofs  = [m[1] for m in cand_models]
tests = [m[2] for m in cand_models]

prob_mat = np.column_stack(oofs)
prob_te  = np.column_stack(tests)
logit_mat = logit(np.clip(prob_mat, 1e-6, 1.0 - 1e-6))
logit_te  = logit(np.clip(prob_te, 1e-6, 1.0 - 1e-6))

print("=== Active Base Model OOF AUCs ===")
for n, p in zip(names, oofs):
    print(f"  {n:16s}: {roc_auc_score(y, p):.5f}")

# 1) Static Logit & Prob Blend Baseline
rng = np.random.default_rng(42)
W = np.vstack([np.eye(len(oofs)), rng.dirichlet(np.ones(len(oofs)), size=400)])
best_prob_auc, best_prob_w = max(((roc_auc_score(y, prob_mat @ w), w) for w in W), key=lambda t: t[0])
best_log_auc, best_log_w = max(((roc_auc_score(y, logit_mat @ w), w) for w in W), key=lambda t: t[0])

def polish_weights(mat, y_target, w_init, steps=3):
    w = w_init.copy()
    best = roc_auc_score(y_target, mat @ w)
    for delta in [0.04, 0.015, 0.005, 0.002]:
        for _ in range(steps):
            for i in range(len(w)):
                for sign in [-1.0, 1.0]:
                    cand = w.copy()
                    cand[i] = max(0.0, cand[i] + sign * delta)
                    if cand.sum() == 0:
                        continue
                    cand /= cand.sum()
                    score = roc_auc_score(y_target, mat @ cand)
                    if score > best:
                        best = score
                        w = cand
    return best, w

best_prob_auc, best_prob_w = polish_weights(prob_mat, y, best_prob_w)
best_log_auc, best_log_w = polish_weights(logit_mat, y, best_log_w)
static_oof_logit = logit_mat @ best_log_w
static_te_logit  = logit_te @ best_log_w

# 2) Build Context-Aware Meta-Features for 2nd-Level Stacking
def build_meta_features(L_mat, df_ctx):
    idx_tree = [i for i, n in enumerate(names) if n.startswith(("LGB", "XGB", "CB", "Shallow"))]
    idx_glm  = [i for i, n in enumerate(names) if "GLM" in n]
    idx_nn   = [i for i, n in enumerate(names) if n in ("PLR_WideResNet", "PyTorch_MLP")]

    tree_mean = L_mat[:, idx_tree].mean(axis=1, keepdims=True)
    glm_mean  = L_mat[:, idx_glm].mean(axis=1, keepdims=True)
    nn_mean   = L_mat[:, idx_nn].mean(axis=1, keepdims=True)

    diff_tree_glm = L_mat[:, 0:1] - glm_mean
    diff_tree_nn  = L_mat[:, 0:1] - nn_mean
    diff_glm_nn   = glm_mean - nn_mean
    std_all       = L_mat.std(axis=1, keepdims=True)
    max_all       = L_mat.max(axis=1, keepdims=True)
    min_all       = L_mat.min(axis=1, keepdims=True)
    span_all      = max_all - min_all

    ctx_cols = [
        "risk_flag_count", "merchant_risk_score", "velocity_score",
        "amount_usd_log1p", "hours_since_last_txn_log1p",
        "cvv_retry_count", "prior_disputes", "txn_count_last_24h",
        "is_foreign_transaction", "used_vpn", "ip_country_mismatch", "is_ai_generated_scam_attempt"
    ]
    C_raw = df_ctx[ctx_cols].to_numpy(dtype=np.float32, na_value=0.0)

    inter_terms = []
    for k in range(6):
        c_col = C_raw[:, k:k+1]
        inter_terms.append(diff_tree_glm * c_col)
        inter_terms.append(diff_tree_nn * c_col)
        inter_terms.append(L_mat[:, 0:1] * c_col)
    inter_mat = np.hstack(inter_terms)

    return np.hstack([
        L_mat, tree_mean, glm_mean, nn_mean,
        diff_tree_glm, diff_tree_nn, diff_glm_nn,
        std_all, max_all, min_all, span_all,
        C_raw, inter_mat
    ]).astype(np.float32)

M_tr = build_meta_features(logit_mat, X_dig3)
M_te = build_meta_features(logit_te, X_test_dig3)

meta_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=2026)
meta_lr_oof  = np.zeros(len(y), dtype=np.float64)
meta_lr_test = np.zeros(len(X_test_dig3), dtype=np.float64)
meta_lgb_oof  = np.zeros(len(y), dtype=np.float64)
meta_lgb_test = np.zeros(len(X_test_dig3), dtype=np.float64)

y_meta_arr = y.values.astype(np.int32)

for fold, (tr_idx, va_idx) in enumerate(meta_skf.split(M_tr, y_meta_arr)):
    sc = StandardScaler()
    mtr = np.clip(sc.fit_transform(M_tr[tr_idx]), -6.0, 6.0)
    mva = np.clip(sc.transform(M_tr[va_idx]), -6.0, 6.0)
    mte = np.clip(sc.transform(M_te), -6.0, 6.0)

    sp = SplineTransformer(n_knots=5, degree=2, knots="uniform", include_bias=False)
    sp_tr = sp.fit_transform(np.clip(mtr[:, :len(names)], -4.5, 4.5))
    sp_va = sp.transform(np.clip(mva[:, :len(names)], -4.5, 4.5))
    sp_te = sp.transform(np.clip(mte[:, :len(names)], -4.5, 4.5))

    X_m_tr = np.hstack([mtr, sp_tr])
    X_m_va = np.hstack([mva, sp_va])
    X_m_te = np.hstack([mte, sp_te])

    mlr = LogisticRegression(C=0.05, solver="lbfgs", max_iter=300, tol=1e-4)
    mlr.fit(X_m_tr, y_meta_arr[tr_idx])
    meta_lr_oof[va_idx] = mlr.predict_proba(X_m_va)[:, 1]
    meta_lr_test += mlr.predict_proba(X_m_te)[:, 1] / 10.0

    mlgb = lgb.LGBMClassifier(
        n_estimators=1200, learning_rate=0.015, num_leaves=3, max_depth=2,
        min_child_samples=500, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
        reg_alpha=5.0, reg_lambda=25.0, random_state=fold + 500, verbose=-1, n_jobs=-1,
    )
    mlgb.fit(
        M_tr[tr_idx], y_meta_arr[tr_idx],
        eval_set=[(M_tr[va_idx], y_meta_arr[va_idx])],
        eval_metric="auc", callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    meta_lgb_oof[va_idx] = mlgb.predict_proba(M_tr[va_idx])[:, 1]
    meta_lgb_test += mlgb.predict_proba(M_te)[:, 1] / 10.0

meta_lr_auc  = roc_auc_score(y, meta_lr_oof)
meta_lgb_auc = roc_auc_score(y, meta_lgb_oof)

stack_oofs = np.column_stack([
    static_oof_logit,
    logit(np.clip(meta_lr_oof, 1e-6, 1.0 - 1e-6)),
    logit(np.clip(meta_lgb_oof, 1e-6, 1.0 - 1e-6)),
])
stack_tes = np.column_stack([
    static_te_logit,
    logit(np.clip(meta_lr_test, 1e-6, 1.0 - 1e-6)),
    logit(np.clip(meta_lgb_test, 1e-6, 1.0 - 1e-6)),
])

best_stack_auc = -1.0
best_stack_w = np.array([1.0, 0.0, 0.0])
for a in np.linspace(0.0, 1.0, 51):
    for b in np.linspace(0.0, 1.0 - a, 51):
        c = max(0.0, 1.0 - a - b)
        w = np.array([a, b, c])
        auc = roc_auc_score(y, stack_oofs @ w)
        if auc > best_stack_auc:
            best_stack_auc = auc
            best_stack_w = w

print("\n=== Ensemble & Meta-Stacker Comparison (OOF AUC) ===")
print(f"  Static Prob  Blend:              {best_prob_auc:.5f}")
print(f"  Static Logit Blend:              {best_log_auc:.5f}")
print(f"  Meta-LogReg (Context + Spline):  {meta_lr_auc:.5f}")
print(f"  Meta-LGB    (Regime GBDT):       {meta_lgb_auc:.5f}")
print(f"  FINAL Combined Meta-Stacker:     {best_stack_auc:.5f}  (weights: Static={best_stack_w[0]:.2f}, MetaLR={best_stack_w[1]:.2f}, MetaLGB={best_stack_w[2]:.2f})")

print("\nStatic Logit Blend Weights:")
for n, w in zip(names, best_log_w):
    print(f"  {n:16s}: {w:.3f}")

final_test = expit(stack_tes @ best_stack_w)
sub = pd.DataFrame({"id": test["id"], "is_fraud": final_test})
sub.to_csv("submission.csv", index=False)
sub.head()
